In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os
from dtw import dtw
import matplotlib.pyplot as plt
import io
import requests
import json
import time


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
well_sites = r"C:\Users\romin\OneDrive\Groundwater\RpSy Data\Site information for all selected wells.xlsx"
state_boundaries = r"C:\Users\romin\OneDrive\Groundwater\cb_2022_us_state_500k"
recharge_dir = r"C:/Users/romin/OneDrive/Groundwater/RpSy Data"
daymet_dir = "./daymet"

NE_states = [
    "Connecticut", "Maine", "Massachusetts", "New Hampshire",
    "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania",
]

# ============================================================
# Load wells and filter to NE states
# ============================================================

def load_sites(path=well_sites):
    df = pd.read_excel(path)
    df = df.rename(columns={
        "ID": "usgs_id",
        "Lat": "lat",
        "Long": "lon",
        "depth (m)": "depth",
    })
    return df

def select_ne_wells(df_sites, states_shp=state_boundaries):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_states)].set_crs(epsg=4326, allow_override=True)
    gdf_sites = gpd.GeoDataFrame(
        df_sites,
        geometry=gpd.points_from_xy(df_sites["lon"], df_sites["lat"]),
        crs="EPSG:4326",
    )
    joined = gpd.sjoin(gdf_sites, ne[["NAME", "geometry"]], how="inner", predicate="within")
    joined = joined.rename(columns={"NAME": "state"}).drop(columns=["index_right"])
    return joined

df_sites = load_sites()
df_ne_wells = select_ne_wells(df_sites)

In [3]:
def get_start_end_dates(df, date_col="Date"):
    df[date_col] = pd.to_datetime(df[date_col])
    return df[date_col].min(), df[date_col].max() 

start_dates = []
end_dates = []
for wid in df_ne_wells["usgs_id"]:
    df = pd.read_csv(f"{recharge_dir}/{wid}.csv")
    start_date, end_date = get_start_end_dates(df)
    start_dates.append(start_date)
    end_dates.append(end_date)

df_ne_wells["start_date"] = start_dates
df_ne_wells["end_date"] = end_dates
df_ne_wells["record_length"] = (df_ne_wells["end_date"] - df_ne_wells["start_date"]).dt.days / 365.25

In [4]:
eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Wells with >=5 year records: {len(eligible_wells)}")

selected_wells = eligible_wells.sample(n=10, random_state=42)
print(selected_wells[["usgs_id", "record_length", "start_date", "end_date"]])

Wells with >=5 year records: 163
             usgs_id  record_length start_date   end_date
426  425803077151201      18.995209 2003-10-02 2022-09-30
403  421746074180201      15.994524 2006-10-02 2022-09-30
422  424520070562401      13.993155 1985-10-02 1999-09-30
306  404639074230001      12.993840 2009-10-02 2022-09-30
356  414330076280501      23.994524 1998-10-02 2022-09-30
274  400229075104601       9.993155 2012-10-02 2022-09-30
458  444904074455201      19.994524 2002-10-02 2022-09-30
302  404140077354001      16.996578 1999-10-02 2016-09-30
364  415228070554601       7.994524 2000-10-02 2008-09-30
439  434217073010601       5.993155 2016-10-02 2022-09-30


In [5]:
# DTW helper functions
def get_events(values, threshold):
    """Return start/end index pairs for runs of consecutive values > threshold."""
    above = (values > threshold).astype(int)
    padded = np.concatenate(([0], above, [0]))
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    return starts, ends

def precip_recharge_event_table(df, precip_col, recharge_col, alignment, precip_threshold=0.0):
    """Return a table of precipitation and recharge events based on DTW alignment."""
    dates = df.index
    precip_vals = df[precip_col].values
    recharge_vals = df[recharge_col].values
    starts, ends = get_events(precip_vals, precip_threshold)

    rows = []
    prev_recharge_idx = None
    for s, e in zip(starts, ends):
        precip_idx = np.arange(s, e + 1)
        mask = np.isin(alignment.index2, precip_idx)
        recharge_idx = np.unique(alignment.index1[mask])

        if prev_recharge_idx is not None and recharge_idx.size > 0:
            test = recharge_idx > prev_recharge_idx.max()
            recharge_idx = recharge_idx[test]   # dropped silently, no print

        row = {
            "precip_start_date": dates[s],
            "precip_end_date": dates[e],
            "precip_total": precip_vals[precip_idx].sum(),
            "precip_peak": precip_vals[precip_idx].max(),
        }
        if recharge_idx.size == 0:
            row.update({
                "recharge_start_date": pd.NaT,
                "recharge_end_date": pd.NaT,
                "recharge_total": np.nan,
                "recharge_peak": np.nan,
                "n_recharge_days": 0,
            })
        else:
            row.update({
                "recharge_start_date": dates[recharge_idx.min()],
                "recharge_end_date": dates[recharge_idx.max()],
                "recharge_total": recharge_vals[recharge_idx].sum(),
                "recharge_peak": recharge_vals[recharge_idx].max(),
                "n_recharge_days": recharge_idx.size,
            })
        rows.append(row)
        prev_recharge_idx = recharge_idx if recharge_idx.size > 0 else prev_recharge_idx

    return pd.DataFrame(rows)

def check_overlapping_recharge_events(event_table):
    """Check for overlapping recharge events in the event table."""
    sorted_table = event_table.sort_values("recharge_start_date")
    overlaps = []
    for i in range(len(sorted_table) - 1):
        current_end = sorted_table.iloc[i]["recharge_end_date"]
        next_start = sorted_table.iloc[i + 1]["recharge_start_date"]
        if pd.notna(current_end) and pd.notna(next_start) and current_end >= next_start:
            overlap_days = (current_end - next_start).days + 1
            overlaps.append((i, current_end, next_start, overlap_days))
    return overlaps

In [6]:
os.makedirs("dtw_event_tables", exist_ok=True)

eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Total wells with >=5 year records: {len(eligible_wells)}")

def run_dtw_pipeline(site_id, recharge_dir=recharge_dir, daymet_dir=daymet_dir,
                      out_dir="dtw_event_tables",
                      precip_threshold=2.5, min_lag=0, max_lag=5):
    recharge_path = f"{recharge_dir}/{site_id}.csv"
    precip_path = f"{daymet_dir}/{site_id}.csv"

    if not os.path.exists(recharge_path) or not os.path.exists(precip_path):
        return None, "missing file"

    df_recharge = pd.read_csv(recharge_path)
    df_precip = pd.read_csv(precip_path)

    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)

    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    if len(df) < 30:
        return None, "too short"

    recharge = df["RpSy (m)"].values
    precip = df["prcp (mm/day)"].values

    if np.std(recharge) == 0 or np.std(precip) == 0:
        return None, "zero variance"

    recharge_norm = recharge / np.std(recharge)
    precip_norm = precip / np.std(precip)

    def causal_window(iw, jw, query_size, reference_size, min_lag=min_lag, max_lag=max_lag, **kwargs):
        lag = iw - jw
        ok = lag >= min_lag
        if max_lag is not None:
            ok = ok & (lag <= max_lag)
        return ok

    try:
        alignment = dtw(
            recharge_norm, precip_norm,
            step_pattern="symmetric2",
            window_type=causal_window,
            window_args={"min_lag": min_lag, "max_lag": max_lag},
            keep_internals=True,
            open_begin=False,
            open_end=False,
        )
    except Exception as e:
        return None, f"dtw failed: {e}"

    event_table = precip_recharge_event_table(
        df, "prcp (mm/day)", "RpSy (m)", alignment, precip_threshold=precip_threshold
    )

    out_path = f"{out_dir}/{site_id}_dtw_event_table.csv"
    event_table.to_csv(out_path, index=False)

    return event_table, "ok"


all_event_tables = {}
failed_wells = {}

for i, site_id in enumerate(eligible_wells["usgs_id"]):
    site_id = str(site_id)
    print(f"\rProcessing well {i+1}/{len(eligible_wells)}...", end="", flush=True)

    out_path = f"dtw_event_tables/{site_id}_dtw_event_table.csv"
    if os.path.exists(out_path):
        all_event_tables[site_id] = pd.read_csv(out_path)
        continue

    table, status = run_dtw_pipeline(site_id)
    if table is not None:
        all_event_tables[site_id] = table
    else:
        failed_wells[site_id] = status

print(f"\n\nCompleted: {len(all_event_tables)} of {len(eligible_wells)} wells")
print(f"Failed/skipped: {len(failed_wells)}")
if failed_wells:
    from collections import Counter
    reasons = Counter(failed_wells.values())
    print("Failure reasons:", dict(reasons))

Total wells with >=5 year records: 163
Processing well 163/163...

Completed: 163 of 163 wells
Failed/skipped: 0


In [7]:
import matplotlib
matplotlib.use('QtAgg')
import matplotlib.pyplot as plt

def plot_dtw_events(site_id, event_table, recharge_dir=recharge_dir, daymet_dir=daymet_dir):
    df_recharge = pd.read_csv(f"{recharge_dir}/{site_id}.csv")
    df_precip = pd.read_csv(f"{daymet_dir}/{site_id}.csv")
    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)
    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    date_cols = ["recharge_start_date", "recharge_end_date", "precip_start_date", "precip_end_date"]
    event_table[date_cols] = event_table[date_cols].apply(pd.to_datetime)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    color = 'tab:red'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Recharge (m)', color=color)
    recharge_max = df["RpSy (m)"].max()
    ax1.set_ylim(0, recharge_max * 1.5)
    ax1.plot(df.index, df["RpSy (m)"], color=color)
    ax1.tick_params(axis='y', labelcolor=color)
   
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Precipitation (mm)', color=color)
    ax2.plot(df.index, df["prcp (mm/day)"], color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    precip_max = df["prcp (mm/day)"].max()
    ax2.set_ylim(precip_max * 1.5, 0)
    axmax = ax1.get_ylim()[1]
    for _, row in event_table.iterrows():
        if pd.notna(row["recharge_start_date"]) and pd.notna(row["recharge_end_date"]):
            x = [row["recharge_end_date"], row["recharge_start_date"],
                 row["precip_start_date"], row["precip_end_date"]]
            y = [0, 0, axmax, axmax]
            ax1.fill(x, y, color='gray', alpha=0.2)
    ax1.set_title(f"Well {site_id}")
    plt.show()


In [8]:
all_ratios = []

for well_id, table in all_event_tables.items():
    df = table.dropna(subset=["precip_total", "recharge_total"]).copy()
    df = df[df["precip_total"] >= 10]  # magnitude floor, testing the ratio cutoff separately
    df["implied_ratio"] = (df["recharge_total"] * 1000) / df["precip_total"]
    df["well_id"] = well_id
    all_ratios.append(df[["well_id", "precip_total", "recharge_total", "implied_ratio"]])

all_ratios_df = pd.concat(all_ratios, ignore_index=True)

print(all_ratios_df["implied_ratio"].describe())
print("\nPercentiles:")
for p in [50, 75, 90, 90, 97.5, 99, 99.5, 99.9]:
    print(f"  {p}th: {all_ratios_df['implied_ratio'].quantile(p/100):.2f}")

count    102828.000000
mean          6.598049
std          13.428459
min           0.000000
25%           0.896236
50%           2.832505
75%           7.029045
max         593.564680
Name: implied_ratio, dtype: float64

Percentiles:
  50th: 2.83
  75th: 7.03
  90th: 14.96
  90th: 14.96
  97.5th: 37.79
  99th: 61.63
  99.5th: 84.88
  99.9th: 154.00


In [9]:
well_max_ratio = all_ratios_df.groupby("well_id")["implied_ratio"].max().reset_index()
well_max_ratio.columns = ["well_id", "max_ratio"]

worst_wells_df = well_max_ratio.sort_values("max_ratio", ascending=False).head(10)
print(worst_wells_df)

             well_id   max_ratio
14   394430077225001  593.564680
51   404140077354001  546.712383
99   414640077493801  358.026329
54   404556077525101  332.798354
84   413026076352901  290.356930
41   402512074414301  269.795204
73   411833075133601  202.445255
56   404708076070701  181.983303
75   412020079133901  169.518855
153  443647070552303  161.569315


In [10]:
for well_id in worst_wells_df["well_id"].head(10):
    print(f"\n{'='*60}\nWell {well_id}\n{'='*60}")
    plot_dtw_events(well_id, all_event_tables[well_id])


Well 394430077225001

Well 404140077354001

Well 414640077493801

Well 404556077525101

Well 413026076352901

Well 402512074414301

Well 411833075133601

Well 404708076070701

Well 412020079133901

Well 443647070552303


In [11]:
MIN_MAG = 5  # minimum precipitation event size (mm) to include

# ============================================================
# LORENZ CURVE — one variable at a time, events sorted smallest to largest
# ============================================================

def lorenz_curve(values):
    """
    x = cumulative fraction of events (smallest to largest),
    y = cumulative fraction of the total. Returns x, y, Gini.
    """
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cum_vals = np.cumsum(sorted_vals)
    cum_frac = cum_vals / cum_vals[-1]
    x = np.concatenate(([0], np.arange(1, n + 1) / n))
    y = np.concatenate(([0], cum_frac))
    trapezoid_func = getattr(np, "trapezoid", None) or np.trapz
    gini = 1 - 2 * trapezoid_func(y, x)
    return x, y, gini


def plot_lorenz(event_table, well_id, value_col, label, min_mag=MIN_MAG,
                  precip_col="precip_total"):
    """
    Plots the Lorenz curve for one variable (precip or recharge),
    only including events with precipitation >= min_mag.
    """
    df = event_table.dropna(subset=[precip_col, value_col]).copy()
    df = df[df[precip_col] >= min_mag]

    if len(df) < 2:
        print(f"Too few events for {well_id}")
        return None

    x, y, gini = lorenz_curve(df[value_col].values)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, color="black", linewidth=2)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6, label="Perfect equality")
    ax.fill_between(x, y, x, alpha=0.15, color="tab:red")
    ax.set_xlabel(f"Cumulative fraction of {label} events (smallest to largest)")
    ax.set_ylabel(f"Cumulative fraction of total {label}")
    ax.set_title(f"Well {well_id} — {label} Lorenz Curve\nGini = {gini:.3f} (n={len(df)}, min={min_mag}mm)")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
    return gini


In [12]:
# ============================================================
# CUMULATIVE PRECIP vs RECHARGE — both variables together
# ============================================================

def plot_cumulative_precip_recharge(event_table, well_id, min_mag=MIN_MAG,
                                       precip_col="precip_total", recharge_col="recharge_total"):
    """
    Sorts events smallest to largest by precipitation, then plots
    cumulative fraction of total precipitation (x) against
    cumulative fraction of total recharge (y).
    Only includes events with precipitation >= min_mag.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df[df[precip_col] >= min_mag]

    if len(df) == 0:
        print(f"No events found for {well_id}. Skipping.")
        return

    df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)

    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    df["cum_precip_frac"] = df[precip_col].cumsum() / total_precip
    df["cum_recharge_frac"] = df[recharge_col].cumsum() / total_recharge

    x = np.concatenate(([0], df["cum_precip_frac"].values))
    y = np.concatenate(([0], df["cum_recharge_frac"].values))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, marker="o", color="black", markersize=3, label="Cumulative curve")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6, label="1:1 line")
    ax.set_xlabel("Cumulative fraction of total precipitation")
    ax.set_ylabel("Cumulative fraction of total recharge")
    ax.set_title(f"Well {well_id} (n={len(df)}, min={min_mag}mm)")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

In [13]:
def plot_cumulative_before_after(event_table, well_id, min_mag=MIN_MAG,
                                    precip_col="precip_total", recharge_col="recharge_total"):
    """
    Side-by-side comparison: cumulative precip-vs-recharge curve with
    NO filtering, vs. with only events >= min_mag included.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

    for ax, mag_cutoff, title in zip(axes, [0, min_mag], ["Unfiltered (all events)", f"Filtered (≥{min_mag}mm)"]):
        df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= mag_cutoff]

        if len(df) == 0:
            ax.set_title(f"{title} (no events)")
            continue

        df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)

        total_precip = df[precip_col].sum()
        total_recharge = df[recharge_col].sum()
        df["cum_precip_frac"] = df[precip_col].cumsum() / total_precip
        df["cum_recharge_frac"] = df[recharge_col].cumsum() / total_recharge

        x = np.concatenate(([0], df["cum_precip_frac"].values))
        y = np.concatenate(([0], df["cum_recharge_frac"].values))

        ax.plot(x, y, marker="o", color="black", markersize=3)
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6)
        ax.set_xlabel("Cumulative fraction of total precipitation")
        ax.set_ylabel("Cumulative fraction of total recharge")
        ax.set_title(f"{title} (n={len(df)})")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)

    fig.suptitle(f"Well {well_id} — Before vs After {min_mag}mm Filter", y=1.02)
    fig.tight_layout()
    plt.show()

In [14]:
worst_well_id = worst_wells_df["well_id"].iloc[0]
plot_cumulative_before_after(all_event_tables[worst_well_id], worst_well_id, min_mag=5)

In [15]:
def add_season(event_table, date_col="precip_start_date"):
    """
    Adds a season column based on meteorological seasons:
    DJF = Winter, MAM = Spring, JJA = Summer, SON = Fall
    """
    df = event_table.copy()
    df[date_col] = pd.to_datetime(df[date_col])  # ensure it's a real datetime, not text

    month = df[date_col].dt.month
    season_map = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Fall", 10: "Fall", 11: "Fall",
    }
    df["season"] = month.map(season_map)
    return df

In [16]:
def plot_cumulative_by_season(event_table, well_id, min_mag=MIN_MAG,
                                 precip_col="precip_total", recharge_col="recharge_total"):
    """
    One figure, four panels (Winter, Spring, Summer, Fall), each
    showing the cumulative precip-vs-recharge curve for that season.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df[df[precip_col] >= min_mag]
    df = add_season(df)

    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 11))

    for ax, season in zip(axes.flat, seasons):
        season_df = df[df["season"] == season].sort_values(precip_col, ascending=True).reset_index(drop=True)

        if len(season_df) == 0:
            ax.set_title(f"{season} (no events)")
            continue

        total_precip = season_df[precip_col].sum()
        total_recharge = season_df[recharge_col].sum()
        season_df["cum_precip_frac"] = season_df[precip_col].cumsum() / total_precip
        season_df["cum_recharge_frac"] = season_df[recharge_col].cumsum() / total_recharge

        x = np.concatenate(([0], season_df["cum_precip_frac"].values))
        y = np.concatenate(([0], season_df["cum_recharge_frac"].values))

        ax.plot(x, y, marker="o", color="black", markersize=3)
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6)
        ax.set_xlabel("Cumulative fraction of precipitation")
        ax.set_ylabel("Cumulative fraction of recharge")
        ax.set_title(f"{season} (n={len(season_df)})")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)

    fig.suptitle(f"Well {well_id} — Seasonal Cumulative Precip vs Recharge (min={min_mag}mm)", y=1.0)
    fig.tight_layout()
    plt.show()

#### 0, 5, 8, and 9 look the strangest. 

In [118]:
worst_well_id = worst_wells_df["well_id"].iloc[8]
plot_cumulative_by_season(all_event_tables[worst_well_id], worst_well_id)

In [18]:
worst_ids = set(worst_wells_df["well_id"])
normal_well_id = [w for w in all_event_tables.keys() if w not in worst_ids][1]

print(f"Using a 'normal' well: {normal_well_id}")
plot_cumulative_by_season(all_event_tables[normal_well_id], normal_well_id)

Using a 'normal' well: 391145074520401


In [19]:
master_csv = pd.read_csv("master_well_summary.csv")
master_csv["well_id"] = master_csv["well_id"].astype(str)
print(f"Loaded: {len(master_csv)} rows, {len(master_csv.columns)} columns")

Loaded: 163 rows, 27 columns


In [20]:
plot_cumulative_by_season(all_event_tables[site_id], site_id)

In [21]:
from scipy.optimize import curve_fit

def kumaraswamy_cdf(x, a, b):
    """Kumaraswamy CDF: F(x) = 1 - (1 - x^a)^b"""
    return 1 - (1 - np.clip(x, 0, 1) ** a) ** b


def fit_kumaraswamy(x_data, y_data):
    """Fits a and b to match the observed cumulative curve. Returns (a, b) or None."""
    try:
        popt, _ = curve_fit(kumaraswamy_cdf, x_data, y_data, p0=[1, 1], maxfev=5000)
        return popt
    except Exception:
        return None


def plot_kumaraswamy_fit(event_table, well_id, min_mag=MIN_MAG,
                            precip_col="precip_total", recharge_col="recharge_total"):
    """
    Fits a Kumaraswamy curve to this well's cumulative precip-vs-recharge
    curve (smallest to largest), and overlays the fitted curve.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df[df[precip_col] >= min_mag]

    if len(df) < 5:
        print(f"Not enough events for well {well_id}. Skipping.")
        return None

    df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)

    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    x_data = np.concatenate(([0], (df[precip_col].cumsum() / total_precip).values))
    y_data = np.concatenate(([0], (df[recharge_col].cumsum() / total_recharge).values))

    params = fit_kumaraswamy(x_data, y_data)
    if params is None:
        print(f"Fit failed for well {well_id}")
        return None

    a, b = params
    x_smooth = np.linspace(0, 1, 100)
    y_smooth = kumaraswamy_cdf(x_smooth, a, b)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(x_data, y_data, "o", color="black", markersize=3, alpha=0.5, label="Actual cumulative curve")
    ax.plot(x_smooth, y_smooth, color="red", linewidth=2, label=f"Kumaraswamy fit (a={a:.2f}, b={b:.2f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5, label="1:1 line")

    ax.set_xlabel("Cumulative fraction of precipitation")
    ax.set_ylabel("Cumulative fraction of recharge")
    ax.set_title(f"Well {well_id} — Kumaraswamy Fit (n={len(df)}, min={min_mag}mm)")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    return a, b


site_id = "391145074520401"
a, b = plot_kumaraswamy_fit(all_event_tables[site_id], site_id)

In [22]:
def fit_kumaraswamy_curve(event_table, min_mag=MIN_MAG,
                             precip_col="precip_total", recharge_col="recharge_total"):
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df[df[precip_col] >= min_mag]
    if len(df) < 5:
        return None
    df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)
    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    x_data = np.concatenate(([0], (df[precip_col].cumsum() / total_precip).values))
    y_data = np.concatenate(([0], (df[recharge_col].cumsum() / total_recharge).values))
    return fit_kumaraswamy(x_data, y_data)


kuma_results = []
for well_id, table in all_event_tables.items():
    params = fit_kumaraswamy_curve(table)
    if params is not None:
        kuma_results.append({"well_id": well_id, "a": params[0], "b": params[1]})

kuma_df = pd.DataFrame(kuma_results)
print(f"Successfully fit: {len(kuma_df)} of {len(all_event_tables)} wells")
print(kuma_df.describe())

Successfully fit: 163 of 163 wells
                a           b
count  163.000000  163.000000
mean     0.935679    0.999021
std      0.155379    0.163467
min      0.556610    0.602079
25%      0.808885    0.884810
50%      0.936288    0.979729
75%      1.028792    1.110667
max      1.342099    1.580704


In [23]:
def compute_well_summary_minmag(all_event_tables, min_mag=MIN_MAG,
                                   precip_col="precip_total", recharge_col="recharge_total"):
    rows = []
    for well_id, table in all_event_tables.items():
        df = table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= min_mag]

        if len(df) < 2:
            continue

        _, _, precip_gini = lorenz_curve(df[precip_col].values)
        _, _, recharge_gini = lorenz_curve(df[recharge_col].values)

        total_precip = df[precip_col].sum()
        total_recharge = df[recharge_col].sum() * 1000  # to mm

        # Kumaraswamy fit
        df_sorted = df.sort_values(precip_col, ascending=True).reset_index(drop=True)
        x_data = np.concatenate(([0], (df_sorted[precip_col].cumsum() / df_sorted[precip_col].sum()).values))
        y_data = np.concatenate(([0], (df_sorted[recharge_col].cumsum() / df_sorted[recharge_col].sum()).values))
        params = fit_kumaraswamy(x_data, y_data)
        kuma_a, kuma_b = (params[0], params[1]) if params is not None else (None, None)

        rows.append({
            "well_id": well_id,
            "n_events": len(df),
            "precip_gini": precip_gini,
            "recharge_gini": recharge_gini,
            "total_precip_mm": total_precip,
            "total_recharge_mm": total_recharge,
            "recharge_efficiency": total_recharge / total_precip if total_precip > 0 else None,
            "kuma_a": kuma_a,
            "kuma_b": kuma_b,
        })
    return pd.DataFrame(rows)


new_summary = compute_well_summary_minmag(all_event_tables, min_mag=MIN_MAG)
print(f"Wells recomputed: {len(new_summary)}")

Wells recomputed: 163


In [119]:
def plot_all_kumaraswamy_curves(well_ids, all_event_tables, title="Kumaraswamy Fits by Well"):
    """
    One figure, every well's fitted curve overlaid as its own colored line.
    """
    fig, ax = plt.subplots(figsize=(9, 8))
    x_smooth = np.linspace(0, 1, 200)

    try:
        cmap = plt.colormaps.get_cmap("tab20")
    except AttributeError:
        cmap = plt.cm.get_cmap("tab20")

    n = max(len(well_ids), 1)

    for i, well_id in enumerate(well_ids):
        params = fit_kumaraswamy_curve(all_event_tables[well_id])
        if params is None:
            continue

        a, b = params
        y_smooth = kumaraswamy_cdf(x_smooth, a, b)
        color = cmap(i / n)
        ax.plot(x_smooth, y_smooth, color=color, linewidth=1.5,
                label=f"{well_id} (a={a:.2f}, b={b:.2f})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.4, label="1:1 line")

    ax.set_xlabel("Cumulative fraction of precipitation (smallest events first)")
    ax.set_ylabel("Cumulative fraction of recharge")
    ax.set_title(title)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=6, loc="upper left", bbox_to_anchor=(1.02, 1))
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

def classify_shape(a, b):
    if pd.isna(a) or pd.isna(b):
        return None
    if a > 1 and b > 1:
        return "Mid-dominant"       # mid-size storms dominate
    elif a > 1 and b <= 1:
        return "Large-dominant"     # few biggest storms dominate
    elif a <= 1 and b > 1:
        return "Small-dominant"     # small storms dominate
    else:
        return "Dual-dominant"      # both small and large extremes matter, mid-size doesn't

kuma_df["shape"] = kuma_df.apply(lambda row: classify_shape(row["a"], row["b"]), axis=1)
print(kuma_df["shape"].value_counts())

shape
Dual-dominant     64
Small-dominant    48
Large-dominant    26
Mid-dominant      25
Name: count, dtype: int64


In [25]:
new_summary["well_id"] = new_summary["well_id"].astype(str)

cols_to_replace = ["n_events", "precip_gini", "recharge_gini", "total_precip_mm",
                    "total_recharge_mm", "recharge_efficiency", "kuma_a", "kuma_b"]

master_csv = master_csv.drop(columns=[c for c in cols_to_replace if c in master_csv.columns], errors="ignore")
master_csv = master_csv.merge(new_summary, on="well_id", how="left")

# Re-derive kuma_shape from the updated a/b values
master_csv["kuma_shape"] = master_csv.apply(lambda row: classify_shape(row["kuma_a"], row["kuma_b"]), axis=1)

master_csv.to_csv("master_well_summary.csv", index=False)
print(f"Updated and saved: {len(master_csv)} rows, {len(master_csv.columns)} columns")

Updated and saved: 163 rows, 27 columns


In [26]:
deep_early_rise = master_csv[
    (master_csv["kuma_shape"] == "Small-dominant") &
    (master_csv["mean_depth_to_water_m"] > master_csv["mean_depth_to_water_m"].median())
].sort_values("mean_depth_to_water_m", ascending=False)

print(deep_early_rise[["well_id", "mean_depth_to_water_m", "landcover_simple", "mtpi", "clay", "sand", "silt"]])

             well_id  mean_depth_to_water_m landcover_simple  mtpi  clay  \
93   414159070310501              29.196771        Developed     1   5.7   
35   401834074515501              20.549142           Forest     0  21.8   
53   404518077575501              20.472123        Developed   -17  14.5   
85   413346075421301              16.894150           Forest    -5  18.6   
25   400120074265401              15.436183           Barren     3  12.1   
88   413542080245002              15.371659        Developed     3  22.4   
55   404639074230001              13.263529          Wetland    -1   NaN   
94   414159079213601              12.392171           Forest   -29  21.2   
75   412020079133901              11.087284           Forest    15  20.1   
92   414129070361401              10.035569        Developed     0   6.5   
98   414632070014901               9.388223        Developed     0  11.9   
15   394440074593101               8.968305        Developed     0  14.2   
91   4141010

In [27]:
master_csv["kuma_shape"] = master_csv.apply(lambda row: classify_shape(row["kuma_a"], row["kuma_b"]), axis=1)
master_csv.to_csv("master_well_summary.csv", index=False)
print(master_csv["kuma_shape"].value_counts())

kuma_shape
Dual-dominant     64
Small-dominant    48
Large-dominant    26
Mid-dominant      25
Name: count, dtype: int64


In [ ]:
def plot_example_curves_by_shape(master_csv, all_event_tables, min_mag=MIN_MAG,
                                    precip_col="precip_total", recharge_col="recharge_total"):
    categories = ["Small-dominant", "Large-dominant", "Mid-dominant", "Dual-dominant"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 11))

    for ax, category in zip(axes.flat, categories):
        candidates = master_csv[master_csv["kuma_shape"] == category]
        if len(candidates) == 0:
            ax.set_title(f"{category} (no wells)")
            continue

        example_well = candidates.iloc[0]["well_id"]
        a, b = candidates.iloc[0]["kuma_a"], candidates.iloc[0]["kuma_b"]

        table = all_event_tables[example_well]
        df = table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= min_mag]
        df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)

        total_precip = df[precip_col].sum()
        total_recharge = df[recharge_col].sum()
        x_data = np.concatenate(([0], (df[precip_col].cumsum() / total_precip).values))
        y_data = np.concatenate(([0], (df[recharge_col].cumsum() / total_recharge).values))

        x_smooth = np.linspace(0, 1, 100)
        y_smooth = kumaraswamy_cdf(x_smooth, a, b)

        ax.plot(x_data, y_data, "o", color="black", markersize=3, alpha=0.4, label="Actual")
        ax.plot(x_smooth, y_smooth, color="red", linewidth=2, label=f"Fit (a={a:.2f}, b={b:.2f})")
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5)

        ax.set_xlabel("Cumulative fraction of precipitation")
        ax.set_ylabel("Cumulative fraction of recharge")
        ax.set_title(f"{category}\nWell {example_well} (n={len(candidates)} wells in category)")
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)

    fig.suptitle("Example Curve for Each Shape Category", y=1.0)
    fig.tight_layout()
    plt.show()


plot_example_curves_by_shape(master_csv, all_event_tables)

In [63]:
def plot_kuma_quadrants(master_csv):
    """
    Four-quadrant scatter of Kumaraswamy a (x) vs b (y), centered
    on (1,1). Quadrants are labeled directly on the plot.
    """
    df = master_csv.dropna(subset=["kuma_a", "kuma_b"])

    fig, ax = plt.subplots(figsize=(10, 10))

    ax.scatter(df["kuma_a"], df["kuma_b"], color="tab:blue", s=50,
               edgecolor="black", linewidth=0.4, alpha=0.6)

    # Cross-shaped axis lines through the center point (1,1)
    ax.axhline(1, color="black", linewidth=1)
    ax.axvline(1, color="black", linewidth=1)
    ax.plot(1, 1, marker="o", color="red", markersize=8, zorder=5)
    ax.annotate("(1,1)", (1, 1), textcoords="offset points", xytext=(8, 8), fontsize=9, color="red")

    x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
    y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1

    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    # Quadrant labels, placed near each corner
    ax.text(x_max, y_max, "Mid-dominant\n(a>1, b>1)", ha="right", va="top", fontsize=10,
             weight="bold")
    ax.text(x_max, y_min, "Large-dominant\n(a>1, b≤1)", ha="right", va="bottom", fontsize=10,
            weight="bold")
    ax.text(x_min, y_max, "Small-dominant\n(a≤1, b>1)", ha="left", va="top", fontsize=10,
             weight="bold")
    ax.text(x_min, y_min, "Dual-dominant\n(a≤1, b≤1)", ha="left", va="bottom", fontsize=10,
             weight="bold")

    ax.set_xlabel("Kumaraswamy a")
    ax.set_ylabel("Kumaraswamy b")
    ax.set_title("Kumaraswamy Shape Parameters")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants(master_csv)

In [64]:
def plot_kuma_quadrants_by_landcover(master_csv):
    """
    Four-quadrant scatter of Kumaraswamy a (x) vs b (y), centered
    on (1,1), colored by land cover type.
    """
    df = master_csv.dropna(subset=["kuma_a", "kuma_b", "landcover_simple"])

    fig, ax = plt.subplots(figsize=(11, 10))

    categories = sorted(df["landcover_simple"].unique())
    cmap = plt.colormaps.get_cmap("tab10")

    for i, category in enumerate(categories):
        subset = df[df["landcover_simple"] == category]
        ax.scatter(subset["kuma_a"], subset["kuma_b"], color=cmap(i / max(len(categories), 1)),
                   s=60, edgecolor="black", linewidth=0.4, alpha=0.75,
                   label=f"{category} (n={len(subset)})")

    ax.axhline(1, color="black", linewidth=1)
    ax.axvline(1, color="black", linewidth=1)
    ax.plot(1, 1, marker="o", color="red", markersize=8, zorder=5)
    ax.annotate("(1,1)", (1, 1), textcoords="offset points", xytext=(8, 8), fontsize=9, color="red")

    x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
    y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1

    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.text(x_max, y_max, "Mid-dominant", ha="right", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_max, y_min, "Large-dominant", ha="right", va="bottom", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_max, "Small-dominant", ha="left", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_min, "Dual-dominant", ha="left", va="bottom", fontsize=10, weight="bold", alpha=0.6)

    ax.set_xlabel("Kumaraswamy a")
    ax.set_ylabel("Kumaraswamy b")
    ax.set_title("Kumaraswamy Shape Parameters by Land Cover")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_landcover(master_csv)

In [65]:
print(f"Range: {master_csv['mean_depth_to_water_m'].min():.2f}m to {master_csv['mean_depth_to_water_m'].max():.2f}m")

Range: 0.46m to 40.63m


In [66]:
def bin_depth(depth):
    if pd.isna(depth):
        return None
    if depth < 4:
        return "0-4m"
    elif depth < 8:
        return "4-8m"
    elif depth < 12:
        return "8-12m"
    elif depth < 16:
        return "12-16m"
    elif depth < 20:
        return "16-20m"
    elif depth < 24:
        return "20-24m"
    elif depth < 28:
        return "24-28m"
    elif depth < 32:
        return "28-32m"
    elif depth < 36:
        return "32-36m"
    elif depth < 40:
        return "36-40m"
    else:
        return "40m+"

def plot_kuma_quadrants_by_depth(master_csv):
    """
    Four-quadrant scatter of Kumaraswamy a (x) vs b (y), centered
    on (1,1), colored by depth-to-water-table bin.
    """
    df = master_csv.dropna(subset=["kuma_a", "kuma_b", "mean_depth_to_water_m"]).copy()
    df["depth_bin"] = df["mean_depth_to_water_m"].apply(bin_depth)

    fig, ax = plt.subplots(figsize=(11, 10))

    bin_order = ["0-4m", "4-8m", "8-12m", "12-16m", "16-20m", "20-24m",
                 "24-28m", "28-32m", "32-36m", "36-40m", "40m+"]

    # Distinct, easily distinguishable colors (tab20 has 20 unique colors)
    cmap = plt.colormaps.get_cmap("tab20")
    colors = [cmap(i) for i in range(len(bin_order))]

    for i, bin_label in enumerate(bin_order):
        subset = df[df["depth_bin"] == bin_label]
        if len(subset) == 0:
            continue
        ax.scatter(subset["kuma_a"], subset["kuma_b"], color=colors[i],
                   s=60, edgecolor="black", linewidth=0.4, alpha=0.8,
                   label=f"{bin_label} (n={len(subset)})")

    ax.axhline(1, color="black", linewidth=1)
    ax.axvline(1, color="black", linewidth=1)
    ax.plot(1, 1, marker="o", color="red", markersize=8, zorder=5)
    ax.annotate("(1,1)", (1, 1), textcoords="offset points", xytext=(8, 8), fontsize=9, color="red")

    x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
    y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1

    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.text(x_max, y_max, "Mid-dominant", ha="right", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_max, y_min, "Large-dominant", ha="right", va="bottom", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_max, "Small-dominant", ha="left", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_min, "Dual-dominant", ha="left", va="bottom", fontsize=10, weight="bold", alpha=0.6)

    ax.set_xlabel("Kumaraswamy a")
    ax.set_ylabel("Kumaraswamy b")
    ax.set_title("Kumaraswamy Shape Parameters by Depth to Water Table, All 163 Wells")
    ax.legend(title="Depth to water table", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_depth(master_csv)

In [67]:
import inspect
print(inspect.getsource(classify_shape))

def classify_shape(a, b):
    if pd.isna(a) or pd.isna(b):
        return None
    if a > 1 and b > 1:
        return "Mid-dominant"       # mid-size storms dominate
    elif a > 1 and b <= 1:
        return "Large-dominant"     # few biggest storms dominate
    elif a <= 1 and b > 1:
        return "Small-dominant"     # small storms dominate
    else:
        return "Dual-dominant"      # both small and large extremes matter, mid-size doesn't



In [68]:
print(master_csv[["kuma_a", "kuma_b"]].describe())

           kuma_a      kuma_b
count  163.000000  163.000000
mean     0.935679    0.999021
std      0.155379    0.163467
min      0.556610    0.602079
25%      0.808885    0.884810
50%      0.936288    0.979729
75%      1.028792    1.110667
max      1.342099    1.580704


#### Build all_filters_summary -- computes Gini/Kumaraswamy a,b for every well under No filter / 5mm / 10mm (this was missing -- the category-count table below depends on it)


In [69]:
def compute_all_filters_summary(all_event_tables):
    """
    For every well, computes precip Gini, recharge Gini, and
    Kumaraswamy a/b under no filter, 5mm filter, and 10mm filter.
    """
    rows = []

    for well_id, table in all_event_tables.items():
        for min_mag, label in [(0, "No filter"), (5, "5mm filter"), (10, "10mm filter")]:
            df = table.dropna(subset=["precip_total", "recharge_total"]).copy()
            df = df[df["precip_total"] >= min_mag]

            if len(df) < 2:
                rows.append({"well_id": well_id, "filter": label, "n_events": len(df),
                             "precip_gini": None, "recharge_gini": None, "kuma_a": None, "kuma_b": None})
                continue

            _, _, precip_gini = lorenz_curve(df["precip_total"].values)
            _, _, recharge_gini = lorenz_curve(df["recharge_total"].values)

            df_sorted = df.sort_values("precip_total", ascending=True).reset_index(drop=True)
            total_precip = df_sorted["precip_total"].sum()
            total_recharge = df_sorted["recharge_total"].sum()
            x_data = np.concatenate(([0], (df_sorted["precip_total"].cumsum() / total_precip).values))
            y_data = np.concatenate(([0], (df_sorted["recharge_total"].cumsum() / total_recharge).values))

            params = fit_kumaraswamy(x_data, y_data)
            a, b = (params[0], params[1]) if params is not None else (None, None)

            rows.append({
                "well_id": well_id, "filter": label, "n_events": len(df),
                "precip_gini": precip_gini, "recharge_gini": recharge_gini,
                "kuma_a": a, "kuma_b": b,
            })

    return pd.DataFrame(rows)


all_filters_summary = compute_all_filters_summary(all_event_tables)
print(f"Total rows: {len(all_filters_summary)}")


Total rows: 489


In [70]:
def classify_by_ab(a, b):
    if pd.isna(a) or pd.isna(b):
        return None
    if a > 1 and b > 1:
        return "a>1, b>1"
    elif a > 1 and b <= 1:
        return "a>1, b≤1"
    elif a <= 1 and b > 1:
        return "a≤1, b>1"
    else:
        return "a≤1, b≤1"


all_filters_summary["ab_condition"] = all_filters_summary.apply(
    lambda row: classify_by_ab(row["kuma_a"], row["kuma_b"]), axis=1
)

category_names = {
    "a>1, b>1": "(Mid-dominant)",
    "a>1, b≤1": "(Large-dominant)",
    "a≤1, b>1": "(Small-dominant)",
    "a≤1, b≤1": "(Dual-dominant)",
}

pivot = all_filters_summary.groupby(["ab_condition", "filter"]).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=["No filter", "5mm filter", "10mm filter"])
pivot["Category"] = pivot.index.map(category_names)
pivot = pivot.reset_index()

print(pivot.to_string(index=False))

ab_condition  No filter  5mm filter  10mm filter         Category
    a>1, b>1         10          25           34   (Mid-dominant)
    a>1, b≤1         20          26           35 (Large-dominant)
    a≤1, b>1         51          48           53 (Small-dominant)
    a≤1, b≤1         82          64           41  (Dual-dominant)


## Kumaraswamy Shape Category Counts by Filter Level

| a/b Condition | Category | No Filter | 5mm Filter | 10mm Filter |
|---|---|---|---|---|
| a>1, b>1 | Mid-dominant | 10 | 25 | 34 |
| a>1, b≤1 | Large-dominant | 20 | 26 | 35 |
| a≤1, b>1 | Small-dominant | 51 | 48 | 53 |
| a≤1, b≤1 | Dual-dominant | 82 | 64 | 41 |
| **Total Wells** | | **163** | **163** | **163** |

In [71]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

filters = ["No filter", "5mm filter", "10mm filter"]

for ax, filt in zip(axes[0], filters):
    data = all_filters_summary[all_filters_summary["filter"] == filt]["kuma_a"].dropna()
    ax.hist(data, bins=25, color="tab:orange", alpha=0.7)
    ax.axvline(1, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("a")
    ax.set_ylabel("Number of wells")
    ax.set_title(f"'a' distribution: {filt}")
    ax.grid(alpha=0.3)

for ax, filt in zip(axes[1], filters):
    data = all_filters_summary[all_filters_summary["filter"] == filt]["kuma_b"].dropna()
    ax.hist(data, bins=25, color="tab:blue", alpha=0.7)
    ax.axvline(1, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("b")
    ax.set_ylabel("Number of wells")
    ax.set_title(f"'b' distribution: {filt}")
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [72]:
def classify_by_ab(a, b):
    if a > 1 and b > 1:
        return "a>1, b>1"
    elif a > 1 and b <= 1:
        return "a>1, b≤1"
    elif a <= 1 and b > 1:
        return "a≤1, b>1"
    else:
        return "a≤1, b≤1"
def compute_kuma_for_threshold(all_event_tables, min_mag,
                                  precip_col="precip_total", recharge_col="recharge_total"):
    rows = []
    for well_id, table in all_event_tables.items():
        df = table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= min_mag]
        if len(df) < 5:
            continue

        df = df.sort_values(precip_col, ascending=True).reset_index(drop=True)
        total_precip = df[precip_col].sum()
        total_recharge = df[recharge_col].sum()
        x_data = np.concatenate(([0], (df[precip_col].cumsum() / total_precip).values))
        y_data = np.concatenate(([0], (df[recharge_col].cumsum() / total_recharge).values))

        params = fit_kumaraswamy(x_data, y_data)
        if params is not None:
            rows.append({"well_id": well_id, "kuma_a": params[0], "kuma_b": params[1]})

    return pd.DataFrame(rows)


kuma_10mm = compute_kuma_for_threshold(all_event_tables, min_mag=10)
kuma_10mm["ab_condition"] = kuma_10mm.apply(lambda row: classify_by_ab(row["kuma_a"], row["kuma_b"]), axis=1)

print(f"Wells fit at 10mm threshold: {len(kuma_10mm)}")
print(kuma_10mm["ab_condition"].value_counts())

Wells fit at 10mm threshold: 163
ab_condition
a≤1, b>1    53
a≤1, b≤1    41
a>1, b≤1    35
a>1, b>1    34
Name: count, dtype: int64


In [73]:
kuma_10mm["ab_condition"] = kuma_10mm.apply(lambda row: classify_by_ab(row["kuma_a"], row["kuma_b"]), axis=1)

print(f"Wells fit at 10mm threshold: {len(kuma_10mm)}")
print(kuma_10mm["ab_condition"].value_counts())

Wells fit at 10mm threshold: 163
ab_condition
a≤1, b>1    53
a≤1, b≤1    41
a>1, b≤1    35
a>1, b>1    34
Name: count, dtype: int64


In [74]:
counts_10mm = kuma_10mm["ab_condition"].value_counts()

comparison_table = pd.DataFrame([
    {"a/b condition": "a>1, b>1",  "Category": "(Mid-dominant)",
     "Ratio filter + 10mm filter": 34, "5mm filter": 25, "10mm filter": counts_10mm.get("a>1, b>1", 0)},
    {"a/b condition": "a>1, b≤1",  "Category": "(Large-dominant)",
     "Ratio filter + 10mm filter": 68, "5mm filter": 26, "10mm filter": counts_10mm.get("a>1, b≤1", 0)},
    {"a/b condition": "a≤1, b>1",  "Category": "(Small-dominant)",
     "Ratio filter + 10mm filter": 18, "5mm filter": 48, "10mm filter": counts_10mm.get("a≤1, b>1", 0)},
    {"a/b condition": "a≤1, b≤1",  "Category": "(Dual-dominant)",
     "Ratio filter + 10mm filter": 43, "5mm filter": 64, "10mm filter": counts_10mm.get("a≤1, b≤1", 0)},
])

print(comparison_table.to_string(index=False))

a/b condition         Category  Ratio filter + 10mm filter  5mm filter  10mm filter
     a>1, b>1   (Mid-dominant)                          34          25           34
     a>1, b≤1 (Large-dominant)                          68          26           35
     a≤1, b>1 (Small-dominant)                          18          48           53
     a≤1, b≤1  (Dual-dominant)                          43          64           41


In [75]:
# Compute a/b for both thresholds (5mm and 10mm), per well
kuma_5mm = compute_kuma_for_threshold(all_event_tables, min_mag=5)
kuma_10mm = compute_kuma_for_threshold(all_event_tables, min_mag=10)

kuma_5mm["well_id"] = kuma_5mm["well_id"].astype(str)
kuma_10mm["well_id"] = kuma_10mm["well_id"].astype(str)

comparison = kuma_5mm.merge(kuma_10mm, on="well_id", suffixes=("_5mm", "_10mm"))

comparison["a_change"] = comparison["kuma_a_10mm"] - comparison["kuma_a_5mm"]
comparison["b_change"] = comparison["kuma_b_10mm"] - comparison["kuma_b_5mm"]

print(f"Wells compared: {len(comparison)}")
print("\nHow much did 'a' change between 5mm and 10mm filtering?")
print(comparison["a_change"].describe())

print("\nHow much did 'b' change between 5mm and 10mm filtering?")
print(comparison["b_change"].describe())

print(f"\nAverage absolute change in a: {comparison['a_change'].abs().mean():.4f}")
print(f"Average absolute change in b: {comparison['b_change'].abs().mean():.4f}")

Wells compared: 163

How much did 'a' change between 5mm and 10mm filtering?
count    163.000000
mean       0.044572
std        0.065243
min       -0.199600
25%        0.004983
50%        0.044000
75%        0.084622
max        0.232917
Name: a_change, dtype: float64

How much did 'b' change between 5mm and 10mm filtering?
count    163.000000
mean       0.017524
std        0.045746
min       -0.160383
25%       -0.003303
50%        0.020681
75%        0.042495
max        0.172139
Name: b_change, dtype: float64

Average absolute change in a: 0.0629
Average absolute change in b: 0.0370


In [76]:
kuma_unfiltered = compute_kuma_for_threshold(all_event_tables, min_mag=0)
print(f"Wells fit with no filter: {len(kuma_unfiltered)}")
print(kuma_unfiltered[["kuma_a", "kuma_b"]].describe())

Wells fit with no filter: 163
           kuma_a      kuma_b
count  163.000000  163.000000
mean     0.868273    0.966073
std      0.163053    0.167245
min      0.523454    0.586267
25%      0.740356    0.847242
50%      0.856353    0.939448
75%      0.977123    1.066925
max      1.374423    1.594194


In [77]:
kuma_unfiltered["ab_condition"] = kuma_unfiltered.apply(lambda row: classify_by_ab(row["kuma_a"], row["kuma_b"]), axis=1)
counts_unfiltered = kuma_unfiltered["ab_condition"].value_counts()

comparison_table = pd.DataFrame([
    {"a/b condition": "a>1, b>1",  "Category": "(Mid-dominant)",
     "No filter": counts_unfiltered.get("a>1, b>1", 0),
     "Ratio filter": 34, "5mm filter": 25, "10mm filter": 34},
    {"a/b condition": "a>1, b≤1",  "Category": "(Large-dominant)",
     "No filter": counts_unfiltered.get("a>1, b≤1", 0),
     "Ratio filter": 68, "5mm filter": 26, "10mm filter": 35},
    {"a/b condition": "a≤1, b>1",  "Category": "(Small-dominant)",
     "No filter": counts_unfiltered.get("a≤1, b>1", 0),
     "Ratio filter": 18, "5mm filter": 48, "10mm filter": 53},
    {"a/b condition": "a≤1, b≤1",  "Category": "(Dual-dominant)",
     "No filter": counts_unfiltered.get("a≤1, b≤1", 0),
     "Ratio filter": 43, "5mm filter": 64, "10mm filter": 41},
])

print(comparison_table.to_string(index=False))

a/b condition         Category  No filter  Ratio filter  5mm filter  10mm filter
     a>1, b>1   (Mid-dominant)         10            34          25           34
     a>1, b≤1 (Large-dominant)         20            68          26           35
     a≤1, b>1 (Small-dominant)         51            18          48           53
     a≤1, b≤1  (Dual-dominant)         82            43          64           41


In [78]:
def plot_gini_comparison(master_csv):
    """
    Precipitation Gini (x) vs Recharge Gini (y), all 163 wells.
    Points above the 1:1 line mean recharge is MORE concentrated
    than precipitation at that well.
    """
    df = master_csv.dropna(subset=["precip_gini", "recharge_gini"])

    fig, ax = plt.subplots(figsize=(10, 10))

    ax.scatter(df["precip_gini"], df["recharge_gini"], color="tab:blue", s=55,
               edgecolor="black", linewidth=0.4, alpha=0.7)

    x_min, x_max = df["precip_gini"].min(), df["precip_gini"].max()
    y_min, y_max = df["recharge_gini"].min(), df["recharge_gini"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    line_min = min(x_min - pad_x, y_min - pad_y)
    line_max = max(x_max + pad_x, y_max + pad_y)

    ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="gray", alpha=0.6, label="1:1 line")

    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.text(line_max * 0.98, line_max * 0.85, "Recharge MORE\nconcentrated\nthan precip",
            ha="right", va="top", fontsize=10, weight="bold", alpha=0.6, color="tab:red")
    ax.text(line_max * 0.85, line_min + (line_max - line_min) * 0.05, "Recharge LESS\nconcentrated\nthan precip",
            ha="right", va="bottom", fontsize=10, weight="bold", alpha=0.6, color="tab:green")

    ax.set_xlabel("Precipitation Gini")
    ax.set_ylabel("Recharge Gini")
    ax.set_title("Precipitation vs Recharge Concentration")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_gini_comparison(master_csv)

C:\Users\romin\AppData\Local\Temp\ipykernel_37584\332308176.py:35: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


In [79]:
def plot_gini_quadrants(master_csv):
    df = master_csv.dropna(subset=["precip_gini", "recharge_gini"])

    precip_median = df["precip_gini"].median()
    recharge_median = df["recharge_gini"].median()

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.scatter(df["precip_gini"], df["recharge_gini"], color="tab:blue", s=55,
               edgecolor="black", linewidth=0.4, alpha=0.7)

    x_min, x_max = df["precip_gini"].min(), df["precip_gini"].max()
    y_min, y_max = df["recharge_gini"].min(), df["recharge_gini"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    line_min = min(x_min - pad_x, y_min - pad_y)
    line_max = max(x_max + pad_x, y_max + pad_y)

    ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="gray", alpha=0.6, label="1:1 line")

    ax.axvline(precip_median, color="black", linewidth=1)
    ax.axhline(recharge_median, color="black", linewidth=1)
    ax.plot(precip_median, recharge_median, marker="o", color="red", markersize=8, zorder=5)
    ax.annotate(f"median\n({precip_median:.2f}, {recharge_median:.2f})", (precip_median, recharge_median),
                textcoords="offset points", xytext=(10, 10), fontsize=9, color="red")

    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.set_xlabel("Precipitation Gini")
    ax.set_ylabel("Recharge Gini")
    ax.set_title("Precipitation vs Recharge Concentration, Split at Median")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_gini_quadrants(master_csv)

In [80]:
physio_path = r"C:\Users\romin\Downloads\physio_shp\physio.shp"

physio = gpd.read_file(physio_path)
print(f"Loaded: {len(physio)} polygons")
print(physio.columns.tolist())
print(physio.head())

Loaded: 501 polygons
['AREA', 'PERIMETER', 'PHYSIODD_', 'PHYSIODD_I', 'FCODE', 'FENCODE', 'DIVISION', 'PROVINCE', 'SECTION', 'PROVCODE', 'geometry']
     AREA  PERIMETER  PHYSIODD_  PHYSIODD_I  FCODE FENCODE  \
0  40.121     36.938          2          72    122     12b   
1  21.976     39.951          3          59    131     13a   
2   2.706     18.014          4          33    241     24a   
3   3.636      8.140          5          34    231     23a   
4  30.059     28.208          6          48    190      19   

                  DIVISION                  PROVINCE  \
0          INTERIOR PLAINS           CENTRAL LOWLAND   
1          INTERIOR PLAINS              GREAT PLAINS   
2  PACIFIC MOUNTAIN SYSTEM            PACIFIC BORDER   
3  PACIFIC MOUNTAIN SYSTEM  CASCADE-SIERRA MOUNTAINS   
4    ROCKY MOUNTAIN SYSTEM  NORTHERN ROCKY MOUNTAINS   

                       SECTION  PROVCODE  \
0                 WESTERN LAKE        12   
1  MISSOURI PLATEAU, GLACIATED        13   
2        

In [81]:
physio = gpd.read_file(physio_path)
print(f"Loaded: {len(physio)} polygons")
print(physio.columns.tolist())

def assign_physiographic_province(wells_df, physio_gdf):
    """
    Spatially joins each well to its physiographic province and division,
    using the same CRS workaround as the state boundary join
    (to_crs() has been unreliable on this environment).
    """
    physio_clean = physio_gdf.copy()
    physio_clean["geometry"] = physio_clean["geometry"].buffer(0)
    physio_clean = physio_clean.set_crs(epsg=4326, allow_override=True)

    gdf_wells = gpd.GeoDataFrame(
        wells_df,
        geometry=gpd.points_from_xy(wells_df["lon"], wells_df["lat"]),
        crs="EPSG:4326",
    )

    joined = gpd.sjoin(gdf_wells, physio_clean[["PROVINCE", "DIVISION", "SECTION", "geometry"]],
                        how="left", predicate="within")
    joined = joined.drop(columns=["index_right"], errors="ignore")
    return joined


wells_with_physio = assign_physiographic_province(master_csv, physio)
print(f"Wells matched to a province: {wells_with_physio['PROVINCE'].notna().sum()} of {len(wells_with_physio)}")
print(wells_with_physio["PROVINCE"].value_counts())

Loaded: 501 polygons
['AREA', 'PERIMETER', 'PHYSIODD_', 'PHYSIODD_I', 'FCODE', 'FENCODE', 'DIVISION', 'PROVINCE', 'SECTION', 'PROVCODE', 'geometry']
Wells matched to a province: 163 of 163
PROVINCE
NEW ENGLAND             46
APPALACHIAN PLATEAUS    34
COASTAL PLAIN           31
VALLEY AND RIDGE        23
PIEDMONT                14
CENTRAL LOWLAND          9
ST. LAWRENCE VALLEY      4
BLUE RIDGE               2
Name: count, dtype: int64


In [82]:
master_csv["well_id"] = master_csv["well_id"].astype(str)
wells_with_physio["well_id"] = wells_with_physio["well_id"].astype(str)

master_csv = master_csv.drop(columns=["PROVINCE", "DIVISION", "SECTION"], errors="ignore")
master_csv = master_csv.merge(
    wells_with_physio[["well_id", "PROVINCE", "DIVISION", "SECTION"]],
    on="well_id", how="left"
)

master_csv.to_csv("master_well_summary.csv", index=False)

print(f"Master CSV updated: {len(master_csv)} rows, {len(master_csv.columns)} columns")
print(master_csv.columns.tolist())

Master CSV updated: 163 rows, 27 columns
['well_id', 'mtpi', 'elevation_m', 'mean_depth_to_water_m', 'clay', 'sand', 'silt', 'state', 'lat', 'lon', 'nlcd_code', 'landcover', 'landcover_simple', 'kuma_shape', 'n_events_x', 'n_events_y', 'n_events', 'precip_gini', 'recharge_gini', 'total_precip_mm', 'total_recharge_mm', 'recharge_efficiency', 'kuma_a', 'kuma_b', 'PROVINCE', 'DIVISION', 'SECTION']


In [83]:
def plot_kuma_quadrants_by_province(master_csv):
    """
    Four-quadrant scatter of Kumaraswamy a (x) vs b (y), centered
    on (1,1), colored by physiographic province.
    """
    df = master_csv.dropna(subset=["kuma_a", "kuma_b", "PROVINCE"])

    fig, ax = plt.subplots(figsize=(11, 10))

    categories = sorted(df["PROVINCE"].unique())
    cmap = plt.colormaps.get_cmap("tab10")

    for i, category in enumerate(categories):
        subset = df[df["PROVINCE"] == category]
        ax.scatter(subset["kuma_a"], subset["kuma_b"], color=cmap(i / max(len(categories), 1)),
                   s=60, edgecolor="black", linewidth=0.4, alpha=0.75,
                   label=f"{category} (n={len(subset)})")

    ax.axhline(1, color="black", linewidth=1)
    ax.axvline(1, color="black", linewidth=1)
    ax.plot(1, 1, marker="o", color="red", markersize=8, zorder=5)
    ax.annotate("(1,1)", (1, 1), textcoords="offset points", xytext=(8, 8), fontsize=9, color="red")

    x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
    y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.text(x_max, y_max, "Mid-dominant", ha="right", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_max, y_min, "Large-dominant", ha="right", va="bottom", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_max, "Small-dominant", ha="left", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_min, "Dual-dominant", ha="left", va="bottom", fontsize=10, weight="bold", alpha=0.6)

    ax.set_xlabel("Kumaraswamy a")
    ax.set_ylabel("Kumaraswamy b")
    ax.set_title("Kumaraswamy Shape Parameters by Physiographic Province")
    ax.legend(title="Physiographic Province", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_province(master_csv)

In [84]:
def plot_gini_comparison_by_province(master_csv):
    """
    Precipitation Gini (x) vs Recharge Gini (y), colored by
    physiographic province.
    """
    df = master_csv.dropna(subset=["precip_gini", "recharge_gini", "PROVINCE"])

    fig, ax = plt.subplots(figsize=(11, 10))

    categories = sorted(df["PROVINCE"].unique())
    cmap = plt.colormaps.get_cmap("tab10")

    for i, category in enumerate(categories):
        subset = df[df["PROVINCE"] == category]
        ax.scatter(subset["precip_gini"], subset["recharge_gini"], color=cmap(i / max(len(categories), 1)),
                   s=60, edgecolor="black", linewidth=0.4, alpha=0.75,
                   label=f"{category} (n={len(subset)})")

    x_min, x_max = df["precip_gini"].min(), df["precip_gini"].max()
    y_min, y_max = df["recharge_gini"].min(), df["recharge_gini"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    line_min = min(x_min - pad_x, y_min - pad_y)
    line_max = max(x_max + pad_x, y_max + pad_y)

    ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="black", linewidth=1.5)

    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.set_xlabel("Precipitation Gini")
    ax.set_ylabel("Recharge Gini")
    ax.set_title("Precipitation vs Recharge Concentration by Physiographic Province")
    ax.legend(title="Physiographic Province", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_gini_comparison_by_province(master_csv)

In [85]:
def plot_gini_comparison_by_landuse(master_csv):
    """
    Precipitation Gini (x) vs Recharge Gini (y), colored by land use type.
    """
    df = master_csv.dropna(subset=["precip_gini", "recharge_gini", "landcover_simple"])

    fig, ax = plt.subplots(figsize=(11, 10))

    categories = sorted(df["landcover_simple"].unique())
    cmap = plt.colormaps.get_cmap("tab10")

    for i, category in enumerate(categories):
        subset = df[df["landcover_simple"] == category]
        ax.scatter(subset["precip_gini"], subset["recharge_gini"], color=cmap(i / max(len(categories), 1)),
                   s=60, edgecolor="black", linewidth=0.4, alpha=0.75,
                   label=f"{category} (n={len(subset)})")

    x_min, x_max = df["precip_gini"].min(), df["precip_gini"].max()
    y_min, y_max = df["recharge_gini"].min(), df["recharge_gini"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    line_min = min(x_min - pad_x, y_min - pad_y)
    line_max = max(x_max + pad_x, y_max + pad_y)

    ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="black", linewidth=1.5)
    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.set_xlabel("Precipitation Gini")
    ax.set_ylabel("Recharge Gini")
    ax.set_title("Precipitation vs Recharge Concentration by Land Use")
    ax.legend(title="Land Use", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_gini_comparison_by_landuse(master_csv)

In [86]:
print(master_csv["mtpi"].describe())

count    163.000000
mean      -5.644172
std       12.627953
min      -60.000000
25%       -9.500000
50%       -2.000000
75%        0.000000
max       25.000000
Name: mtpi, dtype: float64


In [87]:
def bin_mtpi(value):
    if pd.isna(value):
        return None
    lower = int(np.floor(value / 10) * 10)
    upper = lower + 10
    return f"{lower} to {upper}"


def plot_kuma_quadrants_by_mtpi_bin(master_csv):
    """
    Four-quadrant scatter of Kumaraswamy a (x) vs b (y), centered
    on (1,1), colored by mTPI binned in increments of 10.
    """
    df = master_csv.dropna(subset=["kuma_a", "kuma_b", "mtpi"]).copy()
    df["mtpi_bin"] = df["mtpi"].apply(bin_mtpi)

    fig, ax = plt.subplots(figsize=(11, 10))

    # Sort bins numerically by their lower bound, not alphabetically
    bin_order = sorted(df["mtpi_bin"].unique(), key=lambda x: int(x.split(" to ")[0]))

    cmap = plt.colormaps.get_cmap("rainbow")

    for i, bin_label in enumerate(bin_order):
        subset = df[df["mtpi_bin"] == bin_label]
        ax.scatter(subset["kuma_a"], subset["kuma_b"],
                   color=cmap(i / max(len(bin_order) - 1, 1)),
                   s=60, edgecolor="black", linewidth=0.4, alpha=0.75,
                   label=f"{bin_label} (n={len(subset)})")

    ax.axhline(1, color="black", linewidth=1)
    ax.axvline(1, color="black", linewidth=1)
    ax.plot(1, 1, marker="o", color="black", markersize=8, zorder=5)
    ax.annotate("(1,1)", (1, 1), textcoords="offset points", xytext=(8, 8), fontsize=9)

    x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
    y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.text(x_max, y_max, "Mid-dominant", ha="right", va="top", fontsize=10, weight="bold", alpha=0.5)
    ax.text(x_max, y_min, "Large-dominant", ha="right", va="bottom", fontsize=10, weight="bold", alpha=0.5)
    ax.text(x_min, y_max, "Small-dominant", ha="left", va="top", fontsize=10, weight="bold", alpha=0.5)
    ax.text(x_min, y_min, "Dual-dominant", ha="left", va="bottom", fontsize=10, weight="bold", alpha=0.5)

    ax.set_xlabel("Kumaraswamy a")
    ax.set_ylabel("Kumaraswamy b")
    ax.set_title("Kumaraswamy Shape Parameters by mTPI (bins of 10)")
    ax.legend(title="mTPI bin", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_mtpi_bin(master_csv)

In [88]:
import requests
import io

def get_usgs_groundwater(site_id, start_date="2014-01-01", end_date="2024-01-01"):
    """
    Pulls daily mean depth-to-water-level data for one USGS site.
    parameterCd 72019 = depth to water level, feet below land surface
    """
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "rdb",
        "sites": site_id,
        "startDT": start_date,
        "endDT": end_date,
        "parameterCd": "72019",
        "siteType": "GW",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()

    lines = resp.text.splitlines()
    data_lines = [l for l in lines if not l.startswith("#")]
    if len(data_lines) < 3:
        print(f"No data returned for site {site_id}")
        return None

    df = pd.read_csv(io.StringIO("\n".join(data_lines)), sep="\t")
    df = df.drop(index=0)

    mean_col = [c for c in df.columns if c.endswith("00003")]
    if not mean_col:
        print(f"No mean column found for site {site_id}. Columns were: {list(df.columns)}")
        return None
    mean_col = mean_col[0]

    out = df[["site_no", "datetime", mean_col]].copy()
    out = out.rename(columns={mean_col: "depth_to_water_ft"})
    out["datetime"] = pd.to_datetime(out["datetime"])
    out["depth_to_water_ft"] = pd.to_numeric(out["depth_to_water_ft"], errors="coerce")

    return out

In [91]:
def add_water_table_before_storm(event_table, gw_head_df, date_col="precip_start_date",
                                    depth_col="depth_to_water_ft"):
    """
    For each event, finds the USGS depth-to-water reading from the
    day immediately before the storm started. Converts to meters.
    """
    df = event_table.copy()

    depths = []
    for _, row in df.iterrows():
        target_date = row[date_col] - pd.Timedelta(days=1)
        match = gw_head_df[gw_head_df["date"] <= target_date].sort_values("date")

        if len(match) == 0:
            depths.append(None)
        else:
            depths.append(match.iloc[-1][depth_col])

    df["WT_depth_before_ft"] = depths
    df["WT_depth_before_m"] = df["WT_depth_before_ft"] * 0.3048

    return df
import requests
import io

def get_usgs_groundwater(site_id, start_date="2014-01-01", end_date="2024-01-01"):
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "rdb", "sites": site_id, "startDT": start_date, "endDT": end_date,
        "parameterCd": "72019", "siteType": "GW",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()

    lines = resp.text.splitlines()
    data_lines = [l for l in lines if not l.startswith("#")]
    if len(data_lines) < 3:
        print(f"No data returned for site {site_id}")
        return None

    df = pd.read_csv(io.StringIO("\n".join(data_lines)), sep="\t")
    df = df.drop(index=0)

    mean_col = [c for c in df.columns if c.endswith("00003")]
    if not mean_col:
        print(f"No mean column found for site {site_id}. Columns were: {list(df.columns)}")
        return None
    mean_col = mean_col[0]

    out = df[["site_no", "datetime", mean_col]].copy()
    out = out.rename(columns={mean_col: "depth_to_water_ft"})
    out["datetime"] = pd.to_datetime(out["datetime"])
    out["depth_to_water_ft"] = pd.to_numeric(out["depth_to_water_ft"], errors="coerce")

    return out

In [92]:
all_event_tables[site_id]["precip_start_date"] = pd.to_datetime(all_event_tables[site_id]["precip_start_date"])

gw_head = get_usgs_groundwater(
    site_id,
    start_date=all_event_tables[site_id]["precip_start_date"].min().strftime("%Y-%m-%d"),
    end_date="2024-01-01"
)
gw_head = gw_head.rename(columns={"datetime": "date"}) if gw_head is not None and not gw_head.empty else None

if gw_head is not None:
    enriched = add_water_table_before_storm(all_event_tables[site_id], gw_head, date_col="precip_start_date")
    print(f"Wells with valid water table depth: {enriched['WT_depth_before_m'].notna().sum()}")
else:
    print(f"No head data available for well {site_id}.")

Wells with valid water table depth: 1133


In [93]:
def plot_precip_vs_recharge(event_table, well_id, color_col="DUR", color_label="Duration (days)",
                              min_mag=MIN_MAG):
    df = event_table.dropna(subset=["precip_total", "recharge_total", color_col]).copy()
    df = df[df["precip_total"] >= min_mag]
    df["RECH_mm"] = df["recharge_total"] * 1000
    df = df[df["RECH_mm"] > 0]

    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(df["precip_total"], df["RECH_mm"], c=df[color_col], cmap="plasma",
                          alpha=0.6, s=30, edgecolor="none")
    ax.set_xscale("log")
    ax.set_yscale("log")

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label(color_label)

    ax.set_xlabel("Precipitation (mm, log scale)")
    ax.set_ylabel("Recharge (mm, log scale)")
    ax.set_title(f"Well {well_id} — Precip vs Recharge (log-log), colored by water table depth before storm (m)")
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    plt.show()


plot_precip_vs_recharge(enriched, site_id, color_col="WT_depth_before_m", color_label="Water table depth before storm (m)")

In [94]:
site_id = "391145074520401"
plot_precip_vs_recharge(enriched, site_id, color_col="WT_depth_before_m", color_label="Water table depth before storm (m)")

In [95]:
def plot_kuma_quadrants_by_elevation(master_csv):
    """
    Four-quadrant scatter of Kumaraswamy a (x) vs b (y), centered
    on (1,1), colored by elevation.
    """
    df = master_csv.dropna(subset=["kuma_a", "kuma_b", "elevation_m"])

    fig, ax = plt.subplots(figsize=(11, 10))

    scatter = ax.scatter(df["kuma_a"], df["kuma_b"], c=df["elevation_m"], cmap="viridis",
                          s=60, edgecolor="black", linewidth=0.4, alpha=0.8)

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("Elevation (m)")

    ax.axhline(1, color="black", linewidth=1)
    ax.axvline(1, color="black", linewidth=1)
    ax.plot(1, 1, marker="o", color="lime", markersize=8, zorder=5)
    ax.annotate("(1,1)", (1, 1), textcoords="offset points", xytext=(8, 8), fontsize=9)

    x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
    y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    ax.set_xlim(x_min - pad_x, x_max + pad_x)
    ax.set_ylim(y_min - pad_y, y_max + pad_y)

    ax.text(x_max, y_max, "Mid-dominant", ha="right", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_max, y_min, "Large-dominant", ha="right", va="bottom", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_max, "Small-dominant", ha="left", va="top", fontsize=10, weight="bold", alpha=0.6)
    ax.text(x_min, y_min, "Dual-dominant", ha="left", va="bottom", fontsize=10, weight="bold", alpha=0.6)

    ax.set_xlabel("Kumaraswamy a")
    ax.set_ylabel("Kumaraswamy b")
    ax.set_title("Kumaraswamy Shape Parameters by Elevation, All 163 Wells")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_elevation(master_csv)

In [96]:
def compute_kuma_by_season(all_event_tables, min_mag=MIN_MAG,
                              precip_col="precip_total", recharge_col="recharge_total",
                              date_col="precip_start_date"):
    rows = []

    for well_id, table in all_event_tables.items():
        table[date_col] = pd.to_datetime(table[date_col])
        df = table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= min_mag]
        df = add_season(df, date_col=date_col)

        for season in ["Winter", "Spring", "Summer", "Fall"]:
            season_df = df[df["season"] == season].sort_values(precip_col, ascending=True).reset_index(drop=True)

            if len(season_df) < 5:
                continue

            total_precip = season_df[precip_col].sum()
            total_recharge = season_df[recharge_col].sum()
            x_data = np.concatenate(([0], (season_df[precip_col].cumsum() / total_precip).values))
            y_data = np.concatenate(([0], (season_df[recharge_col].cumsum() / total_recharge).values))

            params = fit_kumaraswamy(x_data, y_data)
            if params is not None:
                rows.append({"well_id": well_id, "season": season, "kuma_a": params[0], "kuma_b": params[1]})

    return pd.DataFrame(rows)


kuma_seasonal = compute_kuma_by_season(all_event_tables)
print(kuma_seasonal["season"].value_counts())

C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: divide by zero encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: invalid value encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: divide by zero encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: invalid value encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: divide by zero encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b


season
Winter    163
Spring    163
Summer    163
Fall      163
Name: count, dtype: int64


In [97]:
def plot_kuma_quadrants_by_season(kuma_seasonal):
    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 13))

    for ax, season in zip(axes.flat, seasons):
        df = kuma_seasonal[kuma_seasonal["season"] == season]

        ax.scatter(df["kuma_a"], df["kuma_b"], color="tab:blue", s=55,
                   edgecolor="black", linewidth=0.4, alpha=0.7)

        ax.axhline(1, color="black", linewidth=1)
        ax.axvline(1, color="black", linewidth=1)
        ax.plot(1, 1, marker="o", color="red", markersize=7, zorder=5)

        if len(df) > 0:
            x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
            y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
            pad_x, pad_y = (x_max - x_min) * 0.1 or 0.1, (y_max - y_min) * 0.1 or 0.1
            ax.set_xlim(x_min - pad_x, x_max + pad_x)
            ax.set_ylim(y_min - pad_y, y_max + pad_y)

            xm, xM = ax.get_xlim()
            ym, yM = ax.get_ylim()

            ax.text(xM, yM, "Mid-dominant", ha="right", va="top", fontsize=9, weight="bold", color="black")
            ax.text(xM, ym, "Large-dominant", ha="right", va="bottom", fontsize=9, weight="bold", color="black")
            ax.text(xm, yM, "Small-dominant", ha="left", va="top", fontsize=9, weight="bold", color="black")
            ax.text(xm, ym, "Dual-dominant", ha="left", va="bottom", fontsize=9, weight="bold", color="black")

        ax.set_xlabel("Kumaraswamy a")
        ax.set_ylabel("Kumaraswamy b")
        ax.set_title(f"{season} (n={len(df)} wells)")
        ax.grid(alpha=0.3)

    fig.suptitle("Kumaraswamy Shape Parameters by Season", y=1.0, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_season(kuma_seasonal)

In [98]:
kuma_seasonal["well_id"] = kuma_seasonal["well_id"].astype(str)
master_csv["well_id"] = master_csv["well_id"].astype(str)

kuma_seasonal = kuma_seasonal.merge(master_csv[["well_id", "PROVINCE"]], on="well_id", how="left")
print(kuma_seasonal["PROVINCE"].value_counts())

PROVINCE
NEW ENGLAND             184
APPALACHIAN PLATEAUS    136
COASTAL PLAIN           124
VALLEY AND RIDGE         92
PIEDMONT                 56
CENTRAL LOWLAND          36
ST. LAWRENCE VALLEY      16
BLUE RIDGE                8
Name: count, dtype: int64


In [99]:
def plot_kuma_quadrants_by_season_province(kuma_seasonal):
    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 14))

    df_all = kuma_seasonal.dropna(subset=["kuma_a", "kuma_b", "PROVINCE"])
    categories = sorted(df_all["PROVINCE"].unique())
    cmap = plt.colormaps.get_cmap("tab10")
    colors = {cat: cmap(i / max(len(categories), 1)) for i, cat in enumerate(categories)}

    for ax, season in zip(axes.flat, seasons):
        df = df_all[df_all["season"] == season]

        for cat in categories:
            subset = df[df["PROVINCE"] == cat]
            if len(subset) == 0:
                continue
            ax.scatter(subset["kuma_a"], subset["kuma_b"], color=colors[cat], s=55,
                       edgecolor="black", linewidth=0.4, alpha=0.75, label=f"{cat} (n={len(subset)})")

        ax.axhline(1, color="black", linewidth=1)
        ax.axvline(1, color="black", linewidth=1)
        ax.plot(1, 1, marker="o", color="red", markersize=7, zorder=5)

        if len(df) > 0:
            x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
            y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
            pad_x, pad_y = (x_max - x_min) * 0.1 or 0.1, (y_max - y_min) * 0.1 or 0.1
            ax.set_xlim(x_min - pad_x, x_max + pad_x)
            ax.set_ylim(y_min - pad_y, y_max + pad_y)

            xm, xM = ax.get_xlim()
            ym, yM = ax.get_ylim()

            ax.text(xM, yM, "Mid-dominant", ha="right", va="top", fontsize=8, weight="bold", color="black")
            ax.text(xM, ym, "Large-dominant", ha="right", va="bottom", fontsize=8, weight="bold", color="black")
            ax.text(xm, yM, "Small-dominant", ha="left", va="top", fontsize=8, weight="bold", color="black")
            ax.text(xm, ym, "Dual-dominant", ha="left", va="bottom", fontsize=8, weight="bold", color="black")

        ax.set_xlabel("Kumaraswamy a")
        ax.set_ylabel("Kumaraswamy b")
        ax.set_title(f"{season} (n={len(df)} wells)")
        ax.grid(alpha=0.3)

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Province", bbox_to_anchor=(1.15, 0.9), loc="upper left", fontsize=8)

    fig.suptitle("Kumaraswamy Shape Parameters by Season, colored by Physiographic Province", y=1.0, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_season_province(kuma_seasonal)

In [100]:
kuma_seasonal = kuma_seasonal.merge(master_csv[["well_id", "landcover_simple"]], on="well_id", how="left")
print(kuma_seasonal["landcover_simple"].value_counts())

landcover_simple
Developed      328
Forest         164
Agriculture     88
Wetland         56
Grassland        8
Barren           4
Shrub            4
Name: count, dtype: int64


In [101]:
def plot_kuma_quadrants_by_season_landuse(kuma_seasonal):
    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 14))

    df_all = kuma_seasonal.dropna(subset=["kuma_a", "kuma_b", "landcover_simple"])
    categories = sorted(df_all["landcover_simple"].unique())
    cmap = plt.colormaps.get_cmap("tab10")
    colors = {cat: cmap(i / max(len(categories), 1)) for i, cat in enumerate(categories)}

    for ax, season in zip(axes.flat, seasons):
        df = df_all[df_all["season"] == season]

        for cat in categories:
            subset = df[df["landcover_simple"] == cat]
            if len(subset) == 0:
                continue
            ax.scatter(subset["kuma_a"], subset["kuma_b"], color=colors[cat], s=55,
                       edgecolor="black", linewidth=0.4, alpha=0.75, label=f"{cat} (n={len(subset)})")

        ax.axhline(1, color="black", linewidth=1)
        ax.axvline(1, color="black", linewidth=1)
        ax.plot(1, 1, marker="o", color="red", markersize=7, zorder=5)

        if len(df) > 0:
            x_min, x_max = df["kuma_a"].min(), df["kuma_a"].max()
            y_min, y_max = df["kuma_b"].min(), df["kuma_b"].max()
            pad_x, pad_y = (x_max - x_min) * 0.1 or 0.1, (y_max - y_min) * 0.1 or 0.1
            ax.set_xlim(x_min - pad_x, x_max + pad_x)
            ax.set_ylim(y_min - pad_y, y_max + pad_y)

            xm, xM = ax.get_xlim()
            ym, yM = ax.get_ylim()

            ax.text(xM, yM, "Mid-dominant", ha="right", va="top", fontsize=8, weight="bold", color="black")
            ax.text(xM, ym, "Large-dominant", ha="right", va="bottom", fontsize=8, weight="bold", color="black")
            ax.text(xm, yM, "Small-dominant", ha="left", va="top", fontsize=8, weight="bold", color="black")
            ax.text(xm, ym, "Dual-dominant", ha="left", va="bottom", fontsize=8, weight="bold", color="black")

        ax.set_xlabel("Kumaraswamy a")
        ax.set_ylabel("Kumaraswamy b")
        ax.set_title(f"{season} (n={len(df)} wells)")
        ax.grid(alpha=0.3)

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Land Use", bbox_to_anchor=(1.15, 0.9), loc="upper left", fontsize=8)

    fig.suptitle("Kumaraswamy Shape Parameters by Season, colored by Land Use", y=1.0, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_kuma_quadrants_by_season_landuse(kuma_seasonal)

In [102]:
def compute_gini_by_season(all_event_tables, min_mag=MIN_MAG,
                              precip_col="precip_total", recharge_col="recharge_total",
                              date_col="precip_start_date"):
    rows = []

    for well_id, table in all_event_tables.items():
        table[date_col] = pd.to_datetime(table[date_col])
        df = table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= min_mag]
        df = add_season(df, date_col=date_col)

        for season in ["Winter", "Spring", "Summer", "Fall"]:
            season_df = df[df["season"] == season]
            if len(season_df) < 2:
                continue

            _, _, precip_gini = lorenz_curve(season_df[precip_col].values)
            _, _, recharge_gini = lorenz_curve(season_df[recharge_col].values)

            rows.append({"well_id": well_id, "season": season,
                         "precip_gini": precip_gini, "recharge_gini": recharge_gini})

    return pd.DataFrame(rows)


gini_seasonal = compute_gini_by_season(all_event_tables)
print(gini_seasonal["season"].value_counts())

season
Winter    163
Spring    163
Summer    163
Fall      163
Name: count, dtype: int64


In [103]:
gini_seasonal["well_id"] = gini_seasonal["well_id"].astype(str)
gini_seasonal = gini_seasonal.merge(master_csv[["well_id", "PROVINCE", "landcover_simple", "mtpi"]],
                                       on="well_id", how="left")

In [104]:
def plot_gini_by_season_categorical(gini_seasonal, color_col, color_title):
    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 14))

    df_all = gini_seasonal.dropna(subset=["precip_gini", "recharge_gini", color_col])
    categories = sorted(df_all[color_col].unique())
    cmap = plt.colormaps.get_cmap("tab10")
    colors = {cat: cmap(i / max(len(categories), 1)) for i, cat in enumerate(categories)}

    x_min, x_max = df_all["precip_gini"].min(), df_all["precip_gini"].max()
    y_min, y_max = df_all["recharge_gini"].min(), df_all["recharge_gini"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    line_min = min(x_min - pad_x, y_min - pad_y)
    line_max = max(x_max + pad_x, y_max + pad_y)

    for ax, season in zip(axes.flat, seasons):
        df = df_all[df_all["season"] == season]

        for cat in categories:
            subset = df[df[color_col] == cat]
            if len(subset) == 0:
                continue
            ax.scatter(subset["precip_gini"], subset["recharge_gini"], color=colors[cat], s=55,
                       edgecolor="black", linewidth=0.4, alpha=0.75, label=f"{cat} (n={len(subset)})")

        ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="gray", alpha=0.6)
        ax.set_xlim(x_min - pad_x, x_max + pad_x)
        ax.set_ylim(y_min - pad_y, y_max + pad_y)
        ax.set_xlabel("Precipitation Gini")
        ax.set_ylabel("Recharge Gini")
        ax.set_title(f"{season} (n={len(df)} wells)")
        ax.grid(alpha=0.3)

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, title=color_title, bbox_to_anchor=(1.15, 0.9), loc="upper left", fontsize=8)

    fig.suptitle(f"Precip vs Recharge Concentration by Season, colored by {color_title}", y=1.0, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_gini_by_season_categorical

<function __main__.plot_gini_by_season_categorical(gini_seasonal, color_col, color_title)>

In [105]:
plot_gini_by_season_categorical(gini_seasonal, "landcover_simple", "Land Use")

In [106]:
def plot_gini_by_season(gini_seasonal):
    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 12))

    df_all = gini_seasonal.dropna(subset=["precip_gini", "recharge_gini"])

    x_min, x_max = df_all["precip_gini"].min(), df_all["precip_gini"].max()
    y_min, y_max = df_all["recharge_gini"].min(), df_all["recharge_gini"].max()
    pad_x, pad_y = (x_max - x_min) * 0.1, (y_max - y_min) * 0.1
    line_min = min(x_min - pad_x, y_min - pad_y)
    line_max = max(x_max + pad_x, y_max + pad_y)

    for ax, season in zip(axes.flat, seasons):
        df = df_all[df_all["season"] == season]

        ax.scatter(df["precip_gini"], df["recharge_gini"], color="tab:blue", s=55,
                   edgecolor="black", linewidth=0.4, alpha=0.7)
        ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="gray", alpha=0.6)

        ax.set_xlim(x_min - pad_x, x_max + pad_x)
        ax.set_ylim(y_min - pad_y, y_max + pad_y)
        ax.set_xlabel("Precipitation Gini")
        ax.set_ylabel("Recharge Gini")
        ax.set_title(f"{season} (n={len(df)} wells)")
        ax.grid(alpha=0.3)

    fig.suptitle("Precip vs Recharge Concentration by Season", y=1.0, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_gini_by_season(gini_seasonal)

In [107]:
def ab_to_vector(a, b):
    """
    Converts (a, b) into a vector from the point (1,1):
      magnitude = how far this well is from perfect uniformity
      angle = which direction it deviates in (degrees, 0-360)
    """
    if pd.isna(a) or pd.isna(b):
        return None, None
    dx = a - 1
    dy = b - 1
    magnitude = np.sqrt(dx**2 + dy**2)
    angle = np.degrees(np.arctan2(dy, dx)) % 360
    return magnitude, angle

In [108]:
def compute_kuma_seasonal_for_filter(all_event_tables, min_mag,
                                        precip_col="precip_total", recharge_col="recharge_total",
                                        date_col="precip_start_date"):
    rows = []
    for well_id, table in all_event_tables.items():
        table[date_col] = pd.to_datetime(table[date_col])
        df = table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= min_mag]
        df = add_season(df, date_col=date_col)

        for season in ["Winter", "Spring", "Summer", "Fall"]:
            season_df = df[df["season"] == season].sort_values(precip_col, ascending=True).reset_index(drop=True)
            if len(season_df) < 5:
                continue
            total_precip = season_df[precip_col].sum()
            total_recharge = season_df[recharge_col].sum()
            x_data = np.concatenate(([0], (season_df[precip_col].cumsum() / total_precip).values))
            y_data = np.concatenate(([0], (season_df[recharge_col].cumsum() / total_recharge).values))
            params = fit_kumaraswamy(x_data, y_data)
            if params is not None:
                rows.append({"well_id": well_id, "season": season, "kuma_a": params[0], "kuma_b": params[1]})
    return pd.DataFrame(rows)


kuma_seasonal_none = compute_kuma_seasonal_for_filter(all_event_tables, min_mag=0)
kuma_seasonal_5mm = compute_kuma_seasonal_for_filter(all_event_tables, min_mag=5)
kuma_seasonal_10mm = compute_kuma_seasonal_for_filter(all_event_tables, min_mag=10)

C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: divide by zero encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: invalid value encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: divide by zero encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: invalid value encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: divide by zero encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romin\AppData\Local\Temp\ipykernel_37584\1786009602.py:5: RuntimeWarning: invalid value encountered in power
  return 1 - (1 - np.clip(x, 0, 1) ** a) ** b
C:\Users\romi

In [109]:
def build_wide_summary(kuma_overall, kuma_seasonal, filter_label):
    row = {"Filter": filter_label,
           "Overall a": kuma_overall["kuma_a"].mean(), "Overall b": kuma_overall["kuma_b"].mean()}
    for season in ["Winter", "Spring", "Summer", "Fall"]:
        season_df = kuma_seasonal[kuma_seasonal["season"] == season]
        row[f"{season} a"] = season_df["kuma_a"].mean()
        row[f"{season} b"] = season_df["kuma_b"].mean()
    return row


wide_table = pd.DataFrame([
    build_wide_summary(kuma_unfiltered, kuma_seasonal_none, "No filter"),
    build_wide_summary(kuma_5mm, kuma_seasonal_5mm, "5mm filter"),
    build_wide_summary(kuma_10mm, kuma_seasonal_10mm, "10mm filter"),
])

col_order = ["Filter", "Overall a", "Overall b", "Winter a", "Winter b",
             "Spring a", "Spring b", "Summer a", "Summer b", "Fall a", "Fall b"]
wide_table = wide_table[col_order]

print(wide_table.round(3).to_string(index=False))

     Filter  Overall a  Overall b  Winter a  Winter b  Spring a  Spring b  Summer a  Summer b  Fall a  Fall b
  No filter      0.868      0.966     0.830     0.998     0.856     0.969     0.989     0.832   0.958   0.967
 5mm filter      0.936      0.999     0.889     1.015     0.923     0.997     1.070     0.868   1.023   1.003
10mm filter      0.980      1.017     0.937     1.022     0.976     1.013     1.132     0.900   1.054   1.024


In [110]:
value_cols = ["Overall a", "Overall b", "Winter a", "Winter b",
              "Spring a", "Spring b", "Summer a", "Summer b", "Fall a", "Fall b"]

pct_table = wide_table.copy()
for col in value_cols:
    base = pct_table[col].iloc[0]
    pct_table[f"% Δ {col}"] = ((pct_table[col] - base) / base * 100).round(1)

print(pct_table.round(3).to_string(index=False))

     Filter  Overall a  Overall b  Winter a  Winter b  Spring a  Spring b  Summer a  Summer b  Fall a  Fall b  % Δ Overall a  % Δ Overall b  % Δ Winter a  % Δ Winter b  % Δ Spring a  % Δ Spring b  % Δ Summer a  % Δ Summer b  % Δ Fall a  % Δ Fall b
  No filter      0.868      0.966     0.830     0.998     0.856     0.969     0.989     0.832   0.958   0.967            0.0            0.0           0.0           0.0           0.0           0.0           0.0           0.0         0.0         0.0
 5mm filter      0.936      0.999     0.889     1.015     0.923     0.997     1.070     0.868   1.023   1.003            7.8            3.4           7.0           1.7           7.8           2.9           8.2           4.3         6.8         3.7
10mm filter      0.980      1.017     0.937     1.022     0.976     1.013     1.132     0.900   1.054   1.024           12.9            5.2          12.8           2.4          14.0           4.6          14.5           8.2        10.1         5.9


## Kumaraswamy Mean a/b by Season, Across Filtering Levels

| Filter | Overall a | Overall b | Winter a | Winter b | Spring a | Spring b | Summer a | Summer b | Fall a | Fall b |
|---|---|---|---|---|---|---|---|---|---|---|
| No filter | 0.868 | 0.966 | 0.830 | 0.998 | 0.856 | 0.969 | 0.989 | 0.832 | 0.958 | 0.967 |
| 5mm filter | 0.936 | 0.999 | 0.889 | 1.015 | 0.923 | 0.997 | 1.070 | 0.868 | 1.023 | 1.003 |
| 10mm filter | 0.980 | 1.017 | 0.937 | 1.022 | 0.976 | 1.013 | 1.132 | 0.900 | 1.054 | 1.024 |

### % Change from No Filter

| Filter | Overall a | Overall b | Winter a | Winter b | Spring a | Spring b | Summer a | Summer b | Fall a | Fall b |
|---|---|---|---|---|---|---|---|---|---|---|
| No filter | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% |
| 5mm filter | +7.8% | +3.4% | +7.1% | +1.7% | +7.8% | +2.9% | +8.2% | +4.3% | +6.8% | +3.7% |
| 10mm filter | +12.9% | +5.2% | +12.9% | +2.4% | +14.0% | +4.5% | +14.5% | +8.2% | +10.0% | +5.9% |

In [111]:
def compute_gini_overall_and_seasonal(all_event_tables, min_mag,
                                          precip_col="precip_total", recharge_col="recharge_total",
                                          date_col="precip_start_date"):
    overall_rows = []
    seasonal_rows = []

    for well_id, table in all_event_tables.items():
        table[date_col] = pd.to_datetime(table[date_col])
        df = table.dropna(subset=[precip_col, recharge_col]).copy()
        df = df[df[precip_col] >= min_mag]

        if len(df) >= 2:
            _, _, precip_gini = lorenz_curve(df[precip_col].values)
            _, _, recharge_gini = lorenz_curve(df[recharge_col].values)
            overall_rows.append({"well_id": well_id, "precip_gini": precip_gini, "recharge_gini": recharge_gini})

        df_season = add_season(df, date_col=date_col)
        for season in ["Winter", "Spring", "Summer", "Fall"]:
            season_df = df_season[df_season["season"] == season]
            if len(season_df) < 2:
                continue
            _, _, p_gini = lorenz_curve(season_df[precip_col].values)
            _, _, r_gini = lorenz_curve(season_df[recharge_col].values)
            seasonal_rows.append({"well_id": well_id, "season": season, "precip_gini": p_gini, "recharge_gini": r_gini})

    return pd.DataFrame(overall_rows), pd.DataFrame(seasonal_rows)


gini_overall_none, gini_seasonal_none = compute_gini_overall_and_seasonal(all_event_tables, min_mag=0)
gini_overall_5mm, gini_seasonal_5mm = compute_gini_overall_and_seasonal(all_event_tables, min_mag=5)
gini_overall_10mm, gini_seasonal_10mm = compute_gini_overall_and_seasonal(all_event_tables, min_mag=10)

In [112]:
def build_gini_wide_summary(gini_overall, gini_seasonal, filter_label, value_col="recharge_gini"):
    row = {"Filter": filter_label, "Overall": gini_overall[value_col].mean()}
    for season in ["Winter", "Spring", "Summer", "Fall"]:
        season_df = gini_seasonal[gini_seasonal["season"] == season]
        row[season] = season_df[value_col].mean()
    return row


gini_wide_table = pd.DataFrame([
    build_gini_wide_summary(gini_overall_none, gini_seasonal_none, "No filter"),
    build_gini_wide_summary(gini_overall_5mm, gini_seasonal_5mm, "5mm filter"),
    build_gini_wide_summary(gini_overall_10mm, gini_seasonal_10mm, "10mm filter"),
])

for col in ["Overall", "Winter", "Spring", "Summer", "Fall"]:
    base = gini_wide_table[col].iloc[0]
    gini_wide_table[f"% Δ {col}"] = ((gini_wide_table[col] - base) / base * 100).round(1)

print(gini_wide_table.round(3).to_string(index=False))

     Filter  Overall  Winter  Spring  Summer  Fall  % Δ Overall  % Δ Winter  % Δ Spring  % Δ Summer  % Δ Fall
  No filter    0.623   0.558   0.575   0.691 0.622          0.0         0.0         0.0         0.0       0.0
 5mm filter    0.594   0.525   0.539   0.668 0.590         -4.7        -6.0        -6.2        -3.4      -5.1
10mm filter    0.552   0.485   0.490   0.628 0.545        -11.5       -13.1       -14.7        -9.1     -12.4


In [113]:
precip_gini_wide_table = pd.DataFrame([
    build_gini_wide_summary(gini_overall_none, gini_seasonal_none, "No filter", value_col="precip_gini"),
    build_gini_wide_summary(gini_overall_5mm, gini_seasonal_5mm, "5mm filter", value_col="precip_gini"),
    build_gini_wide_summary(gini_overall_10mm, gini_seasonal_10mm, "10mm filter", value_col="precip_gini"),
])

for col in ["Overall", "Winter", "Spring", "Summer", "Fall"]:
    base = precip_gini_wide_table[col].iloc[0]
    precip_gini_wide_table[f"% Δ {col}"] = ((precip_gini_wide_table[col] - base) / base * 100).round(1)

print(precip_gini_wide_table.round(3).to_string(index=False))

     Filter  Overall  Winter  Spring  Summer  Fall  % Δ Overall  % Δ Winter  % Δ Spring  % Δ Summer  % Δ Fall
  No filter    0.473   0.432   0.466   0.483 0.487          0.0         0.0         0.0         0.0       0.0
 5mm filter    0.411   0.361   0.402   0.427 0.427        -13.1       -16.5       -13.7       -11.6     -12.3
10mm filter    0.340   0.285   0.327   0.356 0.358        -28.2       -34.0       -29.8       -26.2     -26.6


## Recharge Gini — Sensitivity to Filtering Threshold

| Filter | Overall | Winter | Spring | Summer | Fall |
|---|---|---|---|---|---|
| No filter | 0.623 | 0.558 | 0.575 | 0.691 | 0.622 |
| 5mm filter | 0.594 | 0.525 | 0.539 | 0.668 | 0.590 |
| 10mm filter | 0.552 | 0.485 | 0.490 | 0.628 | 0.545 |

### % Change from No Filter

| Filter | Overall | Winter | Spring | Summer | Fall |
|---|---|---|---|---|---|
| No filter | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% |
| 5mm filter | -4.7% | -6.0% | -6.2% | -3.4% | -5.1% |
| 10mm filter | -11.5% | -13.1% | -14.7% | -9.1% | -12.4% |

## Precipitation Gini — Sensitivity to Filtering Threshold

| Filter | Overall | Winter | Spring | Summer | Fall |
|---|---|---|---|---|---|
| No filter | 0.473 | 0.432 | 0.466 | 0.483 | 0.487 |
| 5mm filter | 0.411 | 0.361 | 0.402 | 0.427 | 0.427 |
| 10mm filter | 0.340 | 0.285 | 0.327 | 0.356 | 0.358 |

### % Change from No Filter

| Filter | Overall | Winter | Spring | Summer | Fall |
|---|---|---|---|---|---|
| No filter | 0.0% | 0.0% | 0.0% | 0.0% | 0.0% |
| 5mm filter | -13.1% | -16.5% | -13.7% | -11.6% | -12.3% |
| 10mm filter | -28.2% | -34.0% | -29.8% | -26.2% | -26.6% |

In [114]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

filters = [
    (gini_overall_none, "No filter"),
    (gini_overall_5mm, "5mm filter"),
    (gini_overall_10mm, "10mm filter"),
]

# Compute shared axis limits across all three filter versions
precip_all = pd.concat([df["precip_gini"] for df, _ in filters])
recharge_all = pd.concat([df["recharge_gini"] for df, _ in filters])

precip_xlim = (precip_all.min(), precip_all.max())
recharge_xlim = (recharge_all.min(), recharge_all.max())

# Compute shared y-axis (max bar height) by building histograms once to check counts
precip_max_count = max(np.histogram(df["precip_gini"], bins=25, range=precip_xlim)[0].max() for df, _ in filters)
recharge_max_count = max(np.histogram(df["recharge_gini"], bins=25, range=recharge_xlim)[0].max() for df, _ in filters)

for ax, (df, title) in zip(axes[0], filters):
    ax.hist(df["precip_gini"], bins=25, range=precip_xlim, color="tab:blue", alpha=0.7)
    ax.set_xlim(precip_xlim)
    ax.set_ylim(0, precip_max_count * 1.1)
    ax.set_xlabel("Precipitation Gini")
    ax.set_ylabel("Number of wells")
    ax.set_title(f"Precip Gini: {title}")
    ax.grid(alpha=0.3)

for ax, (df, title) in zip(axes[1], filters):
    ax.hist(df["recharge_gini"], bins=25, range=recharge_xlim, color="tab:green", alpha=0.7)
    ax.set_xlim(recharge_xlim)
    ax.set_ylim(0, recharge_max_count * 1.1)
    ax.set_xlabel("Recharge Gini")
    ax.set_ylabel("Number of wells")
    ax.set_title(f"Recharge Gini: {title}")
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()